# Quantum double-slit wave inversion

Regenerate the physical matrix and the inverse phases at the original grid and width.

## The inverse response is composed from an explicit field block

The displayed Circuit factory prepares the source field and composes all
192 projector angles with alternating block/adjoint calls. FlagQuantum runs
these Circuit kernels; host model.py supplies the 16-by-16 grid, scatterer,
absorbing boundaries and degree-191 reciprocal approximation. This is not a
single native qfunc inferred from a wave equation.

Inspect spectral approximation, unnormalized projected field and workspace
probability separately. The animation visualizes the declared harmonic cycle,
not a fresh hardware time evolution. Try changing a source only with a matching
classical reference and normalization. The original source scale, all phases
and independent full-state checks remain; high fidelity alone does not measure
postselection cost or establish quantum advantage.


1. Set up local FlagQuantum execution and plot rendering.

In [ ]:
import flagquantum as fq
import math

from examples.synthesis.applications.cfd.double_slit_experiment.quantum_double_slit_experiment.overview.quantum import (
    AUX,
    COMPLEX,
    DEGREE,
    Inverse,
    Model,
    WIDTH,
    Y,
    projector,
    signal,
    X,
)


def qsvt_preview(model: Model, inverse: Inverse) -> fq.Circuit:
    """Preserve all 191 signal calls and the source's original width limit."""
    angles = inverse.phases.phases - math.pi / 2
    angles[0] += math.pi / 4
    angles[-1] += math.pi / 4
    if angles.numel() != DEGREE + 1:
        raise ValueError("Expected exactly 192 inverse angles")
    block = signal(model)
    adjoint = block.adjoint()
    circuit = fq.Circuit(WIDTH, dtype=COMPLEX)
    circuit.x(X[-1]).x(X[-2]).x(X[-3]).x(Y[-2])
    circuit.h(X[0])
    for wire in X[1:]:
        circuit.cx(X[0], wire)
    circuit.h(AUX)
    phase_wires = tuple(range(7))
    circuit.any(*phase_wires, matrix=projector(float(angles[0])))
    for index, angle in enumerate(angles[1:]):
        circuit.compose(adjoint if index % 2 == 0 else block)
        circuit.any(*phase_wires, matrix=projector(float(angle)))
    return circuit.h(AUX)


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.cfd.double_slit_experiment.quantum_double_slit_experiment.overview import (
    workflow,
)
from examples.synthesis.applications.cfd.double_slit_experiment.quantum_double_slit_experiment.overview.main import (
    print_report,
    save_animation,
    save_figures,
)
from examples.synthesis.applications.cfd.double_slit_experiment.quantum_double_slit_experiment.overview.model import (
    DEGREE,
    GRID,
    SOURCES,
)


2. Retain the original spatial resolution, constants, and wave sources.

In [ ]:
print("Grid:", GRID, GRID)
print("Sources:", SOURCES)
assert GRID**2 == 256


3. Solve the absorbing-boundary matrix and run the full native inverse.

In [ ]:
result = workflow.run()
assert tuple(result.model.matrix.shape) == (256, 256)
print("Wave block normalization:", result.model.alpha)


4. Render the reference and postselected wave fields with the source phase normalization.

In [ ]:
figures = Path("notebook-figures")
save_figures(result, figures)
display(Image(filename=str(figures / "wave-fields.png")))


5. Check the mapped Chebyshev reciprocal at the original degree.

In [ ]:
assert DEGREE == 191 and result.inverse.coefficients.numel() == 96
display(Image(filename=str(figures / "inverse-poly.png")))


6. Regenerate all phases and compare with the independent singular-value response.

In [ ]:
assert result.inverse.phases.phases.numel() == 192
assert result.inverse.expected_fidelity >= 0.95
assert result.spectral_error < 1e-10


7. Verify full-state phase, postselection and the physical width constraint.

In [ ]:
assert result.resources.n_qubits <= 18
assert result.full_error < 1e-10 and result.work_probability == 0
print_report(result)


8. Animate the real component over a complete physical phase cycle.

In [ ]:
save_animation(result, figures)
display(Image(filename=str(figures / "wave-cycle.gif"), format="gif"))
